In [ ]:
!pip install -q scikit-learn pyarrow

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score


def find_input(filename, root="/kaggle/input"):
    matches = sorted(Path(root).rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"expected exactly one {filename} under {root}, found {len(matches)}: {matches}"
        )
    return matches[0]


holdout_preds = pd.read_parquet(find_input("final_holdout_predictions.parquet"))
holdout_preds["date"] = pd.to_datetime(holdout_preds["date"])
holdout_preds["residual"] = holdout_preds["sales"] - holdout_preds["forecast"]

print(holdout_preds.shape)
print(holdout_preds.head())

In [ ]:
def control_limit_flags(df, residual_col="residual", center_reference_col=None, std_reference_col=None, k=2.5):
    df = df.copy()
    grouped = df.groupby(["store_nbr", "family"])[residual_col]
    center = grouped.transform("mean") if center_reference_col is None else df[center_reference_col]
    spread = grouped.transform("std") if std_reference_col is None else df[std_reference_col]
    df["control_limit_flag"] = ((df[residual_col] - center).abs() > k * spread).astype(int)
    return df


def build_scaled_iso_features_from_clean_reference(df, clean_reference_df, value_cols):
    keys = ["store_nbr", "family"]
    features = pd.DataFrame(index=df.index)
    for injected_col, clean_col in value_cols.items():
        grouped = clean_reference_df.groupby(keys)[clean_col]
        stats = pd.concat(
            [grouped.mean().rename("mean"), grouped.std().replace(0, np.nan).rename("std")],
            axis=1,
        ).reset_index()
        merged = df[keys].merge(stats, on=keys, how="left")
        features[injected_col] = (
            df[injected_col].to_numpy() - merged["mean"].to_numpy()
        ) / merged["std"].to_numpy()
    return features.fillna(0)


def isolation_forest_flags(fit_features, predict_features=None, contamination=0.05, seed=42):
    fit_array = np.asarray(fit_features, dtype=float)
    model = IsolationForest(contamination=contamination, random_state=seed)
    model.fit(fit_array)
    target = fit_array if predict_features is None else np.asarray(predict_features, dtype=float)
    return (model.predict(target) == -1).astype(int)


holdout_preds = control_limit_flags(holdout_preds)
print(f"Control-limit flagged anomalies: {holdout_preds['control_limit_flag'].sum()} / {len(holdout_preds)}")

iso_features = build_scaled_iso_features_from_clean_reference(
    holdout_preds, holdout_preds,
    value_cols={"residual": "residual", "sales": "sales", "forecast": "forecast"},
)
holdout_preds["isoforest_flag"] = isolation_forest_flags(iso_features)
print(f"IsolationForest flagged anomalies: {holdout_preds['isoforest_flag'].sum()} / {len(holdout_preds)}")

In [ ]:
def inject_synthetic_anomalies(df, n_anomalies=50, spike_multiplier_range=(2.5, 5.0),
                               drop_fraction_range=(0.5, 0.9), seed=42):
    rng = np.random.default_rng(seed)
    df = df.copy().reset_index(drop=True)
    df["is_synthetic_anomaly"] = 0
    df["sales_injected"] = df["sales"].astype(float)

    series_mean = df.groupby(["store_nbr", "family"])["sales"].transform("mean")

    anomaly_idx = rng.choice(df.index, size=min(n_anomalies, len(df)), replace=False)
    for idx in anomaly_idx:
        direction = rng.choice(["spike", "drop"])
        row_sales = df.loc[idx, "sales"]
        row_series_mean = series_mean.loc[idx]
        baseline = row_sales if row_sales > 0 else (row_series_mean if row_series_mean > 0 else df["sales"].mean())
        if direction == "spike":
            multiplier = rng.uniform(*spike_multiplier_range)
            injected_value = baseline * (1 + multiplier)
        else:
            drop_frac = rng.uniform(*drop_fraction_range)
            injected_value = baseline * (1 - drop_frac)
        df.loc[idx, "sales_injected"] = max(0, injected_value)
        df.loc[idx, "is_synthetic_anomaly"] = 1

    df["residual_injected"] = df["sales_injected"] - df["forecast"]
    return df


eval_df = inject_synthetic_anomalies(holdout_preds)

clean_residual_stats = (
    holdout_preds.groupby(["store_nbr", "family"])["residual"]
    .agg(clean_residual_mean="mean", clean_residual_std="std")
    .reset_index()
)
eval_df = eval_df.merge(clean_residual_stats, on=["store_nbr", "family"], how="left")
eval_df = control_limit_flags(
    eval_df, residual_col="residual_injected",
    center_reference_col="clean_residual_mean", std_reference_col="clean_residual_std",
)
eval_df = eval_df.rename(columns={"control_limit_flag": "control_limit_flag_injected"})

iso_features_injected = build_scaled_iso_features_from_clean_reference(
    eval_df, holdout_preds,
    value_cols={"residual_injected": "residual", "sales_injected": "sales", "forecast": "forecast"},
)
eval_df["isoforest_flag_injected"] = isolation_forest_flags(
    iso_features, predict_features=iso_features_injected
)

In [ ]:
metrics = {}
for method in ["control_limit_flag_injected", "isoforest_flag_injected"]:
    y_true = eval_df["is_synthetic_anomaly"]
    y_pred = eval_df[method]
    metrics[method] = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

metrics_df = pd.DataFrame(metrics).T
print(metrics_df)

holdout_preds.to_parquet("/kaggle/working/anomaly_results.parquet", index=False)
metrics_df.to_csv("/kaggle/working/anomaly_eval_metrics.csv")
print("Saved: anomaly_results.parquet, anomaly_eval_metrics.csv")